Bearing RUL Prediction Pipeline

This notebook performs **feature engineering** and **Remaining Useful Life (RUL) label construction** using the **PRONOSTIA/FEMTO-ST bearing dataset**.

Objectives
- Extract meaningful vibration features from bearing sensor data
- Construct RUL labels for supervised learning
- Prepare a clean dataset for deep learning and machine learning models

Selected Features

The following condition-monitoring features are extracted:
- RMS
- Kurtosis
- Crest Factor
- Spectral Entropy
- FFT Band Energy

These features are extracted from both X-axis and Y-axis vibration channels.


## Step 1 — Import Required Libraries

This section imports all required Python libraries for:
- Numerical computation
- Signal processing
- Feature extraction
- File handling
- Dataset creation

In [2]:
# Bearing RUL Prediction Pipeline
# Feature Engineering + RUL Label Construction
# Final features:
# RMS, Kurtosis, Crest Factor, Spectral Entropy, FFT Band Energy, RUL

import os
import numpy as np
import pandas as pd
from scipy.stats import kurtosis

## Step 2 — Define Sampling Frequency

The PRONOSTIA dataset vibration signals are sampled at **25.6 kHz**.

This value is used during frequency-domain feature extraction.


In [3]:
# PRONOSTIA sampling frequency
FS = 25600

## Step 3 — Spectral Entropy Function

Spectral entropy measures the randomness or complexity of the frequency spectrum.

### Interpretation
- Low entropy → stable signal
- High entropy → irregular or damaged bearing behavior

This is useful for identifying bearing degradation.


In [4]:
# Spectral Entropy Function

def spectral_entropy(signal):
    fft_vals = np.fft.rfft(signal)

    power = np.abs(fft_vals) ** 2

    power_norm = power / (np.sum(power) + 1e-12)

    entropy = -np.sum(
        power_norm * np.log2(power_norm + 1e-12)
    )

    entropy_norm = entropy / np.log2(len(power_norm))

    return entropy_norm

## Step 4 — FFT Band Energy Function

This function computes signal energy within a selected frequency range using FFT.

### Why it matters
Bearing faults often generate energy spikes in specific frequency bands.

The selected frequency band:
- Lower limit = 500 Hz
- Upper limit = 5000 Hz

In [5]:
# FFT Band Energy Function

def fft_band_energy(signal, fs=FS, f_low=500, f_high=5000):

    fft_vals = np.fft.rfft(signal)

    freqs = np.fft.rfftfreq(
        len(signal),
        d=1/fs
    )

    power = np.abs(fft_vals) ** 2

    band_mask = (
        (freqs >= f_low) &
        (freqs <= f_high)
    )

    band_energy = np.sum(power[band_mask])

    return band_energy

## Step 5 — Feature Extraction Function

This function:
1. Reads raw vibration CSV files
2. Extracts X-axis and Y-axis vibration signals
3. Computes statistical and frequency-domain features
4. Returns extracted features as a dictionary

The extracted features will later be used for RUL prediction modeling.

In [6]:
# Feature Extraction Function

def extract_features(filepath):

    df = pd.read_csv(filepath, header=None)

    # Vibration channels
    ax = df[4].values
    ay = df[5].values

    features = {}

    for name, sig in [("x", ax), ("y", ay)]:

        rms_value = np.sqrt(
            np.mean(sig ** 2)
        )

        peak_value = np.max(
            np.abs(sig)
        )

        features[f"rms_{name}"] = rms_value

        features[f"kurtosis_{name}"] = kurtosis(sig)

        features[f"crest_{name}"] = (
            peak_value / (rms_value + 1e-12)
        )

        features[f"spectral_entropy_{name}"] = (
            spectral_entropy(sig)
        )

        features[f"fft_band_energy_{name}"] = (
            fft_band_energy(sig)
        )

    return features

## Step 6 — Load Bearing Data and Construct RUL Labels

This section:
- Iterates through all files of a bearing run
- Extracts features for every time step
- Constructs Remaining Useful Life (RUL) labels

### RUL Formula
RUL = Total Files − Current Time Step

This creates supervised labels for training prediction models.


In [7]:
# Load Bearing Data and Construct RUL Labels

def load_bearing(path):

    files = sorted(os.listdir(path))

    records = []

    total_steps = len(files)

    for i, file in enumerate(files):

        filepath = os.path.join(path, file)

        feats = extract_features(filepath)

        # Remaining Useful Life
        feats["RUL"] = total_steps - i - 1

        # Bearing name
        feats["bearing"] = path.split("/")[-1]

        # Time index
        feats["time_step"] = i

        records.append(feats)

    return pd.DataFrame(records)

## Step 7 — Define Training Bearings

These are the bearing runs selected from the PRONOSTIA Learning Set.

Each folder contains vibration signals collected until bearing failure.

In [8]:
# List of bearing data directories for training

learning_bearings = [

    "Learning_set/Bearing1_1",
    "Learning_set/Bearing1_2",

    "Learning_set/Bearing2_1",
    "Learning_set/Bearing2_2",

    "Learning_set/Bearing3_1",
    "Learning_set/Bearing3_2",
]

## Step 8 — Process and Combine All Bearings

This section:
- Processes each bearing individually
- Extracts all features
- Combines them into a single training dataframe

The final dataframe will be used for machine learning and deep learning models.


In [9]:
# Process each bearing and aggregate data

all_data = []

for path in learning_bearings:

    print(f"Processing {path}...")

    bearing_df = load_bearing(path)

    all_data.append(bearing_df)

train_df = pd.concat(
    all_data,
    ignore_index=True
)

Processing Learning_set/Bearing1_1...
Processing Learning_set/Bearing1_2...
Processing Learning_set/Bearing2_1...
Processing Learning_set/Bearing2_2...
Processing Learning_set/Bearing3_1...
Processing Learning_set/Bearing3_2...


## Step 9 — Select Final Features for Modeling

Only the most informative features are selected for training.

### Selected Features
- RMS
- Kurtosis
- Crest Factor
- Spectral Entropy
- FFT Band Energy
- RUL target label

Feature selection helps:
- Reduce overfitting
- Improve model generalization
- Reduce training complexity


In [10]:
# Final feature set for modeling

final_columns = [

    "rms_x",
    "kurtosis_x",
    "crest_x",
    "spectral_entropy_x",
    "fft_band_energy_x",

    "rms_y",
    "kurtosis_y",
    "crest_y",
    "spectral_entropy_y",
    "fft_band_energy_y",

    "RUL",
    "bearing",
    "time_step",
]

train_df = train_df[final_columns]

## Step 10 — Display Dataset Information

This section checks:
- Dataset dimensions
- Feature columns
- Sample records

This helps verify successful feature extraction.

In [11]:
# Display dataset shape and sample records

print(train_df.shape)

train_df.head()

(7534, 13)


,rms_x,kurtosis_x,crest_x,spectral_entropy_x,fft_band_energy_x,rms_y,kurtosis_y,crest_y,spectral_entropy_y,fft_band_energy_y,RUL,bearing,time_step
0,0.561746,-0.131465,3.578132,0.737411,881481.869549,0.435801,-0.035080,3.650745,0.889255,218045.175171,2802,Bearing1_1,0
1,0.535112,-0.084646,3.578687,0.771654,814291.485204,0.420968,0.150620,3.957542,0.897493,218825.543172,2801,Bearing1_1,1
2,0.531158,0.033388,3.578971,0.756740,788306.889011,0.425605,-0.061552,3.721758,0.887962,187232.918954,2800,Bearing1_1,2
3,0.554833,0.043419,3.442476,0.747789,862229.478882,0.445524,-0.113319,3.591275,0.896555,222925.632007,2799,Bearing1_1,3
4,0.566652,-0.185477,3.118317,0.742210,886410.660726,0.423847,-0.034816,3.239377,0.901043,188442.307009,2798,Bearing1_1,4


In [12]:
# Save the processed training features to CSV

train_df.to_csv(
    "new_features_hanna.csv",
    index=False
)

print("Saved new_features_hanna.csv!")

Saved new_features_hanna.csv!
